# Object-embedding batch processing

Embed every `*_semantics.json` file recursively under a configurable list of
folders using [core.dom.embedding.object](01.core.dom.embedding.object.ipynb).
This is the embedding companion to [the object-summary batch notebook](05.application.summary.object.ipynb).

Each document's nonempty `summary` and all recursive object `summary` fields become
vectors in a dedicated Chroma collection. SQLite retains raw content, source IDs,
types, paths, section context, and parent/child links. Mirrored records are indexed
once. Empty or intentionally ignored summaries remain in the catalog without vectors.
Section/prose `s` fields are not additional vector records.

Run cells in order. Opening this notebook does not process corpora. Source JSON,
Markdown, summaries, and summary checkpoints are never modified. Live processing
cells are excluded from automated notebook tests.

## Configuration

Set `INPUT_DIRS` to folders containing existing `*_semantics.json` files; nested
folders are included. Defaults match the outputs of the object-summary application.
Folders are processed in list order, with sorted files within each folder. Overlapping
roots and repeated paths are deduplicated globally; the first folder owns the file.
Missing or empty folders fail discovery before the index is opened.

Optional configuration is loaded from `PROJ_ROOT/.env`, with existing environment
variables taking precedence. `OLLAMA_EMBED_MODEL` defaults to `qwen3-embedding:8b`,
independent of the summarization model. `OLLAMA_EMBED_HOSTS` defaults to the
comma-separated pair `http://localhost:11434,http://172.27.74.16:11434`.
Both services must have the same embedding model installed. `OLLAMA_API_KEY`
supplies optional bearer authentication. `RIBOSOME_EMBED_REQUEST_TIMEOUT` defaults
to 300 seconds including queue wait. `RIBOSOME_EMBED_SERVER_CONCURRENCY` defaults
to four requests per host (eight concurrent requests in total). Faster hosts take more
work as slots become available. Configuration and discovery make no provider
requests and do not create index files.

`INDEX_DIR` defaults to `PROJ_ROOT/db/object_embeddings/qwen3-embedding-8b`, matching
the core notebook. Keep host set, model, and index location consistent when resuming.
Model-specific directories preserve prior `bge-m3` indexes. A provider-set change
requires a separate index directory; changing concurrency alone does not.
`PREVIEW_OUTPUTS` limits inspection only, never
processing. Embedding always resumes safely; there is no checkpoint reset or forced
rebuild toggle.

In [ ]:
# | notest
import asyncio
import json
import math
import os
import sys
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv


def resolve_project_root():
    configured = os.getenv("PROJ_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(f"Invalid PROJ_ROOT (missing pyproject.toml): {root}")
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pyproject.toml").is_file():
            return root
    raise FileNotFoundError("Cannot locate PROJ_ROOT containing pyproject.toml")


PROJ_ROOT = resolve_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

import pandas as pd
from IPython.display import display
from utils.embedding_ollama_pool import OllamaEmbeddingPool
from tqdm.auto import tqdm
from ribosome.core.dom.embedding.object import ObjectEmbeddingIndex

INPUT_DIRS = [
    PROJ_ROOT / "assets" / "code-docs.4.0" / ".md",
    PROJ_ROOT / "assets" / "code-docs.5.0" / ".md",
]
EMBED_MODEL = (os.getenv("OLLAMA_EMBED_MODEL") or "qwen3-embedding:8b").strip()
OLLAMA_HOSTS = list(dict.fromkeys(host.strip().rstrip("/") for host in os.getenv(
    "OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434"
).split(",") if host.strip()))
SERVER_CONCURRENCY = int(os.getenv("RIBOSOME_EMBED_SERVER_CONCURRENCY", "4"))
EMBEDDING_CONCURRENCY = len(OLLAMA_HOSTS) * SERVER_CONCURRENCY
INDEX_DIR = PROJ_ROOT / "db" / "object_embeddings" / EMBED_MODEL.replace("/", "_").replace(":", "-")
CATALOG_PATH = INDEX_DIR / "catalog.sqlite3"
REQUEST_TIMEOUT = float(os.getenv("RIBOSOME_EMBED_REQUEST_TIMEOUT", "300"))
if not OLLAMA_HOSTS or not EMBED_MODEL or SERVER_CONCURRENCY < 1:
    raise ValueError("Embedding hosts/model must be nonempty and server concurrency must be positive")
if not math.isfinite(REQUEST_TIMEOUT) or REQUEST_TIMEOUT <= 0:
    raise ValueError("RIBOSOME_EMBED_REQUEST_TIMEOUT must be finite and positive")
EMBEDDING_CONFIG = {"provider": "ollama-pool", "hosts": sorted(OLLAMA_HOSTS)}
api_key = os.getenv("OLLAMA_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
PREVIEW_OUTPUTS = 3
PROGRESS_INTERVAL = 0.25
print("Inputs:")
for input_dir in INPUT_DIRS:
    print(f"  {input_dir}")
print(f"Embedding hosts: {OLLAMA_HOSTS}\nEmbedding model: {EMBED_MODEL}\nConcurrent requests: {EMBEDDING_CONCURRENCY}")
print(f"Index directory: {INDEX_DIR}\nCatalog: {CATALOG_PATH}")

## Discovery and batch helpers

The document bar shows completed files, the current source path, status counts, and
elapsed time. It refreshes while awaiting the embedding provider. One file is published
at a time, respecting the catalog's single-writer contract. Unique summaries within
each file run concurrently through bounded async workers on both Ollama servers. Individual failures are
reported and other files continue. Cancelling stops and awaits the active task, leaves
prior published generations intact, and retains collected results for inspection.
Synchronous parsing and Chroma writes finish before cancellation can be observed.

In [ ]:
import asyncio
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

from tqdm.auto import tqdm


def discover_semantics(input_dirs):
    """Validate every folder and deduplicate resolved files in configured order."""
    if isinstance(input_dirs, (str, Path)):
        input_dirs = [input_dirs]
    input_dirs = list(input_dirs)
    if not input_dirs:
        raise ValueError("INPUT_DIRS must contain at least one folder")
    corpora, seen = [], set()
    for value in input_dirs:
        folder = Path(value).expanduser().resolve()
        if not folder.is_dir():
            raise FileNotFoundError(f"Semantics input directory does not exist: {folder}")
        files = sorted({path.resolve() for path in folder.rglob("*_semantics.json") if path.is_file()})
        if not files:
            raise ValueError(f"No *_semantics.json files found under: {folder}")
        unique = [path for path in files if path not in seen]
        seen.update(unique)
        corpora.append((folder, unique))
    return corpora


async def embed_corpora(corpora, object_index, results, *, progress=True, progress_interval=0.25):
    """Append one report per finished file; propagate cancellation after task cleanup."""
    if not 0 < progress_interval < float("inf"):
        raise ValueError("progress_interval must be finite and positive")
    counts = {"indexed": 0, "unchanged": 0, "failed": 0}
    total = sum(len(files) for _, files in corpora)
    with tqdm(total=total, desc="Documents", unit="file", disable=not progress) as bar:
        for folder, files in corpora:
            for file in files:
                started = perf_counter()
                _ = bar.set_postfix(file=str(file), **counts)
                task = asyncio.create_task(object_index.index_file(file))
                try:
                    while not task.done():
                        _ = await asyncio.wait({task}, timeout=progress_interval)
                        _ = bar.refresh()
                    result = asdict(await task)
                except Exception as exc:
                    result = {"semantics_file": str(file), "status": "failed",
                              "error": f"{type(exc).__name__}: {exc}"}
                finally:
                    if not task.done():
                        task.cancel()
                        _ = await asyncio.gather(task, return_exceptions=True)
                result.update(corpus=str(folder), elapsed_seconds=perf_counter() - started)
                results.append(result)
                counts[result["status"]] += 1
                _ = bar.set_postfix(file=str(file), **counts)
                _ = bar.update(1)
    return results

## Discover inputs and check the embedding model

Discovery validates all folders before contacting Ollama. Concurrent preflight checks every configured server using
`show`, so installed model tags resolve normally. If the server reports
capabilities, the model must support `embedding` on every host. No embeddings or index writes occur
in this step. On a preflight failure the HTTP client is closed; correct configuration
and rerun this cell before processing.

In [ ]:
# | notest
corpora = discover_semantics(INPUT_DIRS)
for input_dir, files in corpora:
    print(f"{input_dir}: {len(files)} unique semantics files")
print(f"Total unique semantics files: {sum(len(files) for _, files in corpora)}")
if "ollama_client" in globals():
    await ollama_client.aclose()
ollama_client = OllamaEmbeddingPool(hosts=OLLAMA_HOSTS,
    concurrency_per_server=SERVER_CONCURRENCY, headers=headers, request_timeout=REQUEST_TIMEOUT)
try:
    _ = await ollama_client.check_models(EMBED_MODEL)
except BaseException:
    await ollama_client.aclose()
    raise
print(f"Providers ready on all hosts: {EMBED_MODEL}")


## Embed all discovered files

Only this cell opens or creates the persistent index. Every discovered file is
included. Unchanged published documents return `unchanged` without embedding requests.
Changed or interrupted documents reuse cached vectors for identical summaries.
The new generation becomes active only after the complete document is stored.
Failed or cancelled documents preserve their previously active catalog and vectors.

`results` is populated after each file and remains available after interruption.
The catalog and client stay open for inspection below. After an interruption, rerun
this cell to retry; run **Close resources** when finished. Rerunning this cell closes
the previous catalog handle before opening it again. Do not run another writer against
the same catalog. Inactive generations are retained by the core index for later maintenance.

In [ ]:
# | notest
if "object_index" in globals():
    object_index.close()
object_index = ObjectEmbeddingIndex(
    CATALOG_PATH, collection=None, llm_client=ollama_client,
    model=EMBED_MODEL, embedding_config=EMBEDDING_CONFIG,
    embedding_concurrency=EMBEDDING_CONCURRENCY,
)
results = []
batch_start = perf_counter()
try:
    _ = await embed_corpora(corpora, object_index, results, progress_interval=PROGRESS_INTERVAL)
finally:
    batch_elapsed_seconds = perf_counter() - batch_start
    print(f"Batch wall time: {batch_elapsed_seconds:.3f}s; finished files: {len(results)}")

## Results and failures

`indexed` means a document generation was published; `unchanged` means the published
input and configuration already match; `failed` includes a diagnostic. `objects`
counts unique object records excluding the document. `embedded` counts newly computed
vectors and `reused` counts cached vectors (or all nonempty summaries for unchanged
files). `skipped` counts records with empty summaries, including the document if empty.
These counters include document-summary vectors where applicable. Timings describe
this run, including quick unchanged checks. Review failures, then rerun the batch.

In [ ]:
# | notest
results_df = pd.DataFrame(results)
display(results_df)
if not results_df.empty:
    print(results_df.groupby(["corpus", "status"]).size().to_string())
    counts = results_df.reindex(columns=["embedded", "reused", "skipped"]).fillna(0).sum()
    print("Vector/record counts for this run:")
    print(counts.to_string())
    failures = results_df.loc[results_df["status"] == "failed"]
    if not failures.empty:
        print("Failed files:")
        display(failures[["corpus", "semantics_file", "error"]])
print(f"Batch wall time: {batch_elapsed_seconds:.3f}s")
print("Per-server embedding requests:")
display(pd.DataFrame(ollama_client.server_snapshots()))


## Inspect published records

Inspect up to `PREVIEW_OUTPUTS` successful documents, including unchanged ones, through
the public catalog lookup API. Show the complete recursive hierarchy once, including
unembedded/ignored objects. Raw evidence and source locations remain available on each
record; this preview displays a short content excerpt. No model requests or source
writes occur. Set `PREVIEW_OUTPUTS = 0` to disable inspection.

In [ ]:
# | notest
def catalog_preview(index, document_id):
    rows = []
    def visit(identifier, depth=0):
        record = index.get_object(identifier)
        if record is None:
            raise ValueError(f"Missing published catalog record: {identifier}")
        rows.append({
            "id": record.id, "source_id": record.source_id, "parent_id": record.parent_id,
            "children_ids": record.children_ids, "depth": depth, "type": record.type,
            "object_path": record.object_path, "source_file": record.source_file,
            "line_start": record.attributes.get("line_start"),
            "line_end": record.attributes.get("line_end"),
            "summary_status": "embedded" if record.summary.strip() else "catalog only",
            "summary": record.summary, "content_preview": record.content[:200],
        })
        for child_id in record.children_ids:
            visit(child_id, depth + 1)
    visit(document_id)
    return rows


preview_documents = [row for row in results if row["status"] in {"indexed", "unchanged"}][:PREVIEW_OUTPUTS]
for row in preview_documents:
    print(f"Published objects: {row['semantics_file']}")
    preview_df = pd.DataFrame(catalog_preview(object_index, row["document_id"]))
    print(preview_df.groupby(["type", "summary_status"]).size().to_string())
    display(preview_df)

## Close resources and resume later

Close the SQLite catalog and Ollama HTTP client, including after an interrupted batch.
Run provider preflight again before another batch after closing resources. The completed
catalog and vector cache are the embedding resume state; no additional corpus checkpoint
is needed. Never reset the summary notebook's checkpoints to retry embedding.

For semantic search and parent/child evidence expansion, follow the retrieval example
in [the core object embedding notebook](01.core.dom.embedding.object.ipynb) with the same
catalog, model and provider configuration. This notebook does not generate RAG answers.

In [ ]:
# | notest
try:
    if "object_index" in globals():
        object_index.close()
finally:
    if "ollama_client" in globals():
        await ollama_client.aclose()